In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TP3-Retail-Intelligence")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Version Spark :", spark.version)
print("Session Spark créée avec succès !")

26/11/18 14:35:52 WARN Utils: Your hostname, MacBookPro-Fouad resolves to a loopback address: 127.0.0.1; using 10.74.1.164 instead (on interface en0)
26/11/18 14:35:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/18 14:35:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Version Spark : 3.5.6
Session Spark créée avec succès !


In [3]:
from pathlib import Path

# Chemin du fichier source
chemin = Path.home() / "retail-intelligence/tp3/data/input/store_sales.dat"

# Lecture des données avec Spark
df_store_sales = (
    spark.read
    .option("header", False)
    .option("sep", "|")
    .csv(str(chemin))
)

# Afficher les 5 premières lignes
df_store_sales.show(5, truncate=False)

+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+------+-------+-------+-------+-------+------+------+-------+-------+-------+
|_c0  |_c1  |_c2  |_c3  |_c4    |_c5 |_c6  |_c7|_c8|_c9|_c10|_c11 |_c12  |_c13  |_c14   |_c15   |_c16   |_c17   |_c18  |_c19  |_c20   |_c21   |_c22   |
+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+------+-------+-------+-------+-------+------+------+-------+-------+-------+
|36890|71480|10001|44404|493944 |2898|44093|10 |294|0  |31  |18.00|30.78 |23.08 |238.70 |715.48 |558.00 |954.18 |21.46 |0.00  |715.48 |736.94 |157.48 |
|36890|71480|10000|44404|493944 |2898|44093|10 |279|0  |54  |99.21|148.82|83.34 |3535.92|4500.36|5357.34|8036.28|225.02|0.00  |4500.36|4725.38|-856.98|
|36890|71480|15556|44404|493944 |2898|44093|10 |114|0  |39  |79.49|94.59 |83.24 |442.65 |3246.36|3100.11|3689.01|184.07|616.81|2629.55|2813.62|-470.56|
|36890|45392|4597 |89832|1774847|6266|16285|4  |285|1  |20  |74.38|136.12|133.40|54.40  

In [4]:
# 1. Combien de colonnes Spark a-t-il détectées ?
print("Nombre de colonnes :", len(df_store_sales.columns))

# 2. Quels noms et types de colonnes Spark utilise-t-il ?
df_store_sales.printSchema()

# 3. À quoi ressemblent les données ?
df_store_sales.show(5, truncate=False)

# 4. Combien d'enregistrements ont été lus ?
print("Nombre de lignes :", df_store_sales.count())

Nombre de colonnes : 23
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)

+-----+-----+-----+-----+-------+----+-----+---+---+---+----+-----+------+------+-------+-------+-------+-------+------+------+-------+-------+-------+


In [5]:
from pyspark.sql import functions as F

# Compter les valeurs nulles et les chaînes vides
controles = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_store_sales.columns
]

# Afficher les résultats par colonne
df_store_sales.agg(*controles).show(truncate=False)

+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+
|_c0|_c1|_c2|_c3|_c4|_c5|_c6|_c7|_c8|_c9|_c10|_c11|_c12|_c13|_c14|_c15|_c16|_c17|_c18|_c19|_c20|_c21|_c22|
+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+
|0  |0  |0  |0  |238|295|448|0  |158|0  |0   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0   |0   |
+---+---+---+---+---+---+---+---+---+---+----+----+----+----+----+----+----+----+----+----+----+----+----+



In [6]:
from pyspark.sql import functions as F

# Afficher 5 ventes dont la colonne _c4 est manquante
df_store_sales.filter(
    F.col("_c4").isNull() |
    (F.trim(F.col("_c4")) == "")
).select(
    "_c0", "_c2", "_c3", "_c4", "_c5", "_c6", "_c8"
).show(5, truncate=False)

+-----+-----+-----+----+----+-----+---+
|_c0  |_c2  |_c3  |_c4 |_c5 |_c6  |_c8|
+-----+-----+-----+----+----+-----+---+
|36908|13514|42365|NULL|3822|10934|241|
|36908|10738|42365|NULL|3822|10934|149|
|36908|13114|42365|NULL|3822|10934|67 |
|36908|17132|42365|NULL|3822|10934|289|
|36908|16497|42365|NULL|3822|10934|266|
+-----+-----+-----+----+----+-----+---+
only showing top 5 rows



In [8]:
df_store_sales.select(
    "_c0", "_c2", "_c4", "_c8",
    "_c10", "_c20", "_c22"
).show(10, truncate=False)

+-----+-----+-------+---+----+-------+-------+
|_c0  |_c2  |_c4    |_c8|_c10|_c20   |_c22   |
+-----+-----+-------+---+----+-------+-------+
|36890|10001|493944 |294|31  |715.48 |157.48 |
|36890|10000|493944 |279|54  |4500.36|-856.98|
|36890|15556|493944 |114|39  |2629.55|-470.56|
|36890|4597 |1774847|285|20  |2668.00|1180.40|
|36890|3258 |1774847|230|76  |900.60 |153.52 |
|36890|15572|1774847|210|18  |540.54 |-784.44|
|36890|2683 |1774847|151|61  |550.83 |-814.96|
|36890|2699 |1774847|24 |83  |2947.33|341.96 |
|36890|4517 |1774847|242|6   |25.44  |-251.58|
|36890|9638 |1774847|180|5   |9.30   |-22.95 |
+-----+-----+-------+---+----+-------+-------+
only showing top 10 rows



In [9]:
from pyspark.sql import functions as F

df_test_types = df_store_sales.select(
    F.col("_c10").alias("quantite_texte"),
    F.col("_c10").cast("integer").alias("quantite_entier"),
    F.col("_c20").alias("montant_texte"),
    F.col("_c20").cast("decimal(18,2)").alias("montant_decimal"),
    F.col("_c22").alias("benefice_texte"),
    F.col("_c22").cast("decimal(18,2)").alias("benefice_decimal")
)

df_test_types.show(5, truncate=False)
df_test_types.printSchema()

+--------------+---------------+-------------+---------------+--------------+----------------+
|quantite_texte|quantite_entier|montant_texte|montant_decimal|benefice_texte|benefice_decimal|
+--------------+---------------+-------------+---------------+--------------+----------------+
|31            |31             |715.48       |715.48         |157.48        |157.48          |
|54            |54             |4500.36      |4500.36        |-856.98       |-856.98         |
|39            |39             |2629.55      |2629.55        |-470.56       |-470.56         |
|20            |20             |2668.00      |2668.00        |1180.40       |1180.40         |
|76            |76             |900.60       |900.60         |153.52        |153.52          |
+--------------+---------------+-------------+---------------+--------------+----------------+
only showing top 5 rows

root
 |-- quantite_texte: string (nullable = true)
 |-- quantite_entier: integer (nullable = true)
 |-- montant_texte: s

In [10]:
from pyspark.sql import functions as F

# Tester les conversions sur l'ensemble du fichier
df_controle = df_store_sales.select(
    F.col("_c10").alias("quantite_source"),
    F.col("_c10").cast("integer").alias("quantite_convertie"),

    F.col("_c20").alias("montant_source"),
    F.col("_c20").cast("decimal(18,2)").alias("montant_converti"),

    F.col("_c22").alias("benefice_source"),
    F.col("_c22").cast("decimal(18,2)").alias("benefice_converti")
)

# Compter les erreurs de conversion
resultat = df_controle.agg(
    F.sum(
        F.when(
            F.col("quantite_source").isNotNull()
            & F.col("quantite_convertie").isNull(), 1
        ).otherwise(0)
    ).alias("erreurs_quantite"),

    F.sum(
        F.when(
            F.col("montant_source").isNotNull()
            & F.col("montant_converti").isNull(), 1
        ).otherwise(0)
    ).alias("erreurs_montant"),

    F.sum(
        F.when(
            F.col("benefice_source").isNotNull()
            & F.col("benefice_converti").isNull(), 1
        ).otherwise(0)
    ).alias("erreurs_benefice")
)

resultat.show()


+----------------+---------------+----------------+
|erreurs_quantite|erreurs_montant|erreurs_benefice|
+----------------+---------------+----------------+
|               0|              0|               0|
+----------------+---------------+----------------+



In [11]:
# Noms des 23 colonnes du schéma store_sales
noms_colonnes = [
    "ss_sold_date_sk",
    "ss_sold_time_sk",
    "ss_item_sk",
    "ss_customer_sk",
    "ss_cdemo_sk",
    "ss_hdemo_sk",
    "ss_addr_sk",
    "ss_store_sk",
    "ss_promo_sk",
    "ss_ticket_number",
    "ss_quantity",
    "ss_wholesale_cost",
    "ss_list_price",
    "ss_sales_price",
    "ss_ext_discount_amt",
    "ss_ext_sales_price",
    "ss_ext_wholesale_cost",
    "ss_ext_list_price",
    "ss_ext_tax",
    "ss_coupon_amt",
    "ss_net_paid",
    "ss_net_paid_inc_tax",
    "ss_net_profit"
]

# Vérifier que le nombre de noms correspond au fichier
assert len(noms_colonnes) == len(df_store_sales.columns)

# Créer un nouveau DataFrame avec les noms métier
df_store_sales_nomme = df_store_sales.toDF(*noms_colonnes)

# Vérifier le résultat
df_store_sales_nomme.printSchema()

df_store_sales_nomme.select(
    "ss_sold_date_sk",
    "ss_item_sk",
    "ss_quantity",
    "ss_net_paid",
    "ss_net_profit"
).show(5, truncate=False)

root
 |-- ss_sold_date_sk: string (nullable = true)
 |-- ss_sold_time_sk: string (nullable = true)
 |-- ss_item_sk: string (nullable = true)
 |-- ss_customer_sk: string (nullable = true)
 |-- ss_cdemo_sk: string (nullable = true)
 |-- ss_hdemo_sk: string (nullable = true)
 |-- ss_addr_sk: string (nullable = true)
 |-- ss_store_sk: string (nullable = true)
 |-- ss_promo_sk: string (nullable = true)
 |-- ss_ticket_number: string (nullable = true)
 |-- ss_quantity: string (nullable = true)
 |-- ss_wholesale_cost: string (nullable = true)
 |-- ss_list_price: string (nullable = true)
 |-- ss_sales_price: string (nullable = true)
 |-- ss_ext_discount_amt: string (nullable = true)
 |-- ss_ext_sales_price: string (nullable = true)
 |-- ss_ext_wholesale_cost: string (nullable = true)
 |-- ss_ext_list_price: string (nullable = true)
 |-- ss_ext_tax: string (nullable = true)
 |-- ss_coupon_amt: string (nullable = true)
 |-- ss_net_paid: string (nullable = true)
 |-- ss_net_paid_inc_tax: string (n

In [12]:
from pyspark.sql import functions as F

# Les colonnes entières : identifiants et quantité
colonnes_entieres = [
    "ss_sold_date_sk",
    "ss_sold_time_sk",
    "ss_item_sk",
    "ss_customer_sk",
    "ss_cdemo_sk",
    "ss_hdemo_sk",
    "ss_addr_sk",
    "ss_store_sk",
    "ss_promo_sk",
    "ss_ticket_number",
    "ss_quantity"
]

# Les colonnes monétaires
colonnes_decimales = [
    "ss_wholesale_cost",
    "ss_list_price",
    "ss_sales_price",
    "ss_ext_discount_amt",
    "ss_ext_sales_price",
    "ss_ext_wholesale_cost",
    "ss_ext_list_price",
    "ss_ext_tax",
    "ss_coupon_amt",
    "ss_net_paid",
    "ss_net_paid_inc_tax",
    "ss_net_profit"
]

# Créer une nouvelle version typée
df_store_sales_type = df_store_sales_nomme

# Convertir les colonnes entières
for colonne in colonnes_entieres:
    df_store_sales_type = df_store_sales_type.withColumn(
        colonne,
        F.col(colonne).cast("integer")
    )

# Convertir les colonnes décimales
for colonne in colonnes_decimales:
    df_store_sales_type = df_store_sales_type.withColumn(
        colonne,
        F.col(colonne).cast("decimal(18,2)")
    )

# Afficher les types obtenus
df_store_sales_type.printSchema()

root
 |-- ss_sold_date_sk: integer (nullable = true)
 |-- ss_sold_time_sk: integer (nullable = true)
 |-- ss_item_sk: integer (nullable = true)
 |-- ss_customer_sk: integer (nullable = true)
 |-- ss_cdemo_sk: integer (nullable = true)
 |-- ss_hdemo_sk: integer (nullable = true)
 |-- ss_addr_sk: integer (nullable = true)
 |-- ss_store_sk: integer (nullable = true)
 |-- ss_promo_sk: integer (nullable = true)
 |-- ss_ticket_number: integer (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(18,2) (nullable = true)
 |-- ss_list_price: decimal(18,2) (nullable = true)
 |-- ss_sales_price: decimal(18,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(18,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(18,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(18,2) (nullable = true)
 |-- ss_ext_list_price: decimal(18,2) (nullable = true)
 |-- ss_ext_tax: decimal(18,2) (nullable = true)
 |-- ss_coupon_amt: decimal(18,2) (nullable = true)
 |--

In [14]:
from pyspark.sql import functions as F

# Compter les erreurs de conversion pour chaque colonne
expressions = []

for colonne in df_store_sales_nomme.columns:
    erreur = F.sum(
        F.when(
            F.col(f"source.{colonne}").isNotNull()
            & (F.trim(F.col(f"source.{colonne}")) != "")
            & F.col(f"converti.{colonne}").isNull(),
            1
        ).otherwise(0)
    ).alias(colonne)

    expressions.append(erreur)

# Comparer les colonnes source et converties, ligne par ligne
df_comparaison = (
    df_store_sales_nomme.alias("source")
    .withColumn(
        "_position",
        F.monotonically_increasing_id()
    )
)

In [15]:
from pyspark.sql import functions as F

# Compter les erreurs de conversion pour chaque colonne
expressions = []

for colonne in df_store_sales_nomme.columns:
    erreur = F.sum(
        F.when(
            F.col(f"source.{colonne}").isNotNull()
            & (F.trim(F.col(f"source.{colonne}")) != "")
            & F.col(f"converti.{colonne}").isNull(),
            1
        ).otherwise(0)
    ).alias(colonne)

    expressions.append(erreur)

# Comparer les colonnes source et converties, ligne par ligne
df_comparaison = (
    df_store_sales_nomme.alias("source")
    .withColumn(
        "_position",
        F.monotonically_increasing_id()
    )
)

In [16]:
from pyspark.sql import functions as F

controles = []

for colonne in df_store_sales_nomme.columns:
    type_cible = (
        "integer"
        if colonne in colonnes_entieres
        else "decimal(18,2)"
    )

    valeur_source = F.col(colonne)

    conversion = valeur_source.cast(type_cible)

    erreur = F.sum(
        F.when(
            valeur_source.isNotNull()
            & (F.trim(valeur_source) != "")
            & conversion.isNull(),
            1
        ).otherwise(0)
    ).alias(colonne)

    controles.append(erreur)

resultat_controles = df_store_sales_nomme.agg(*controles)

resultat_controles.show(truncate=False)

[Stage 16:=====>                                                   (1 + 9) / 10]

+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+-------------------+-------------+
|ss_sold_date_sk|ss_sold_time_sk|ss_item_sk|ss_customer_sk|ss_cdemo_sk|ss_hdemo_sk|ss_addr_sk|ss_store_sk|ss_promo_sk|ss_ticket_number|ss_quantity|ss_wholesale_cost|ss_list_price|ss_sales_price|ss_ext_discount_amt|ss_ext_sales_price|ss_ext_wholesale_cost|ss_ext_list_price|ss_ext_tax|ss_coupon_amt|ss_net_paid|ss_net_paid_inc_tax|ss_net_profit|
+---------------+---------------+----------+--------------+-----------+-----------+----------+-----------+-----------+----------------+-----------+-----------------+-------------+--------------+-------------------+------------------+---------------------+-----------------+----------+-------------+-----------+

In [17]:
from pyspark.sql import functions as F

# Colonnes candidates pour identifier une ligne de vente
cles_vente = [
    "ss_sold_date_sk",
    "ss_store_sk",
    "ss_ticket_number",
    "ss_item_sk"
]

# Regrouper les ventes ayant la même combinaison de clés
df_groupes = (
    df_store_sales_type
    .groupBy(*cles_vente)
    .count()
)

# Repérer les combinaisons présentes plusieurs fois
df_doublons = df_groupes.filter(F.col("count") > 1)

# Afficher les résultats
print("Nombre de groupes de clés répétés :", df_doublons.count())

df_doublons.show(10, truncate=False)

Nombre de groupes de clés répétés : 4038
+---------------+-----------+----------------+----------+-----+
|ss_sold_date_sk|ss_store_sk|ss_ticket_number|ss_item_sk|count|
+---------------+-----------+----------------+----------+-----+
|37009          |5          |5871            |13530     |2    |
|36928          |10         |1916            |10001     |2    |
|36973          |11         |4126            |5618      |2    |
|36928          |2          |1879            |6575      |2    |
|36937          |10         |2363            |2667      |2    |
|36944          |5          |2673            |7628      |2    |
|37026          |0          |6713            |10000     |2    |
|37026          |10         |6721            |10003     |2    |
|37026          |5          |6723            |10001     |2    |
|37060          |6          |8418            |7612      |2    |
+---------------+-----------+----------------+----------+-----+
only showing top 10 rows



In [18]:
df_store_sales_type.filter(
    (F.col("ss_sold_date_sk") == 37009) &
    (F.col("ss_store_sk") == 5) &
    (F.col("ss_ticket_number") == 5871) &
    (F.col("ss_item_sk") == 13530)
).select(
    "ss_sold_date_sk",
    "ss_store_sk",
    "ss_ticket_number",
    "ss_item_sk",
    "ss_quantity",
    "ss_sales_price",
    "ss_net_paid",
    "ss_net_profit"
).show(truncate=False)

+---------------+-----------+----------------+----------+-----------+--------------+-----------+-------------+
|ss_sold_date_sk|ss_store_sk|ss_ticket_number|ss_item_sk|ss_quantity|ss_sales_price|ss_net_paid|ss_net_profit|
+---------------+-----------+----------------+----------+-----------+--------------+-----------+-------------+
|37009          |5          |5871            |13530     |34         |11.60         |370.74     |-345.30      |
|37009          |5          |5871            |13530     |31         |27.97         |867.07     |-1402.44     |
+---------------+-----------+----------------+----------+-----------+--------------+-----------+-------------+



In [19]:
# Nombre total de lignes
nb_total = df_store_sales_type.count()

# Nombre de lignes distinctes sur les 23 colonnes
nb_distinct = df_store_sales_type.distinct().count()

# Nombre de lignes supplémentaires identiques
nb_doublons_exacts = nb_total - nb_distinct

print("Nombre total de lignes :", nb_total)
print("Nombre de lignes distinctes :", nb_distinct)
print("Nombre de doublons exacts :", nb_doublons_exacts)

[Stage 34:=====>                                                   (1 + 9) / 10]

Nombre total de lignes : 667579
Nombre de lignes distinctes : 667579
Nombre de doublons exacts : 0


In [20]:
from pathlib import Path

chemin_web = Path.home() / "retail-intelligence/tp3/data/input/web_sales.dat"

df_web_sales = (
    spark.read
    .option("header", False)
    .option("sep", "|")
    .csv(str(chemin_web))
)

print("Nombre de colonnes :", len(df_web_sales.columns))
print("Nombre de lignes :", df_web_sales.count())

df_web_sales.printSchema()
df_web_sales.show(3, truncate=False)

Nombre de colonnes : 34
Nombre de lignes : 668052
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (

In [21]:
from pyspark.sql import functions as F

# Construire un contrôle pour chacune des 34 colonnes
controles_web = [
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_web_sales.columns
]

# Calculer le nombre de valeurs manquantes
resultat_web = df_web_sales.agg(*controles_web)

# Afficher uniquement les colonnes avec des valeurs manquantes
valeurs_manquantes_web = resultat_web.first().asDict()

for colonne, nombre in valeurs_manquantes_web.items():
    if nombre > 0:
        print(f"{colonne} : {nombre} valeurs manquantes")

_c5 : 309 valeurs manquantes
_c6 : 288 valeurs manquantes
_c7 : 272 valeurs manquantes
_c9 : 492 valeurs manquantes
_c10 : 495 valeurs manquantes
_c11 : 398 valeurs manquantes
_c12 : 164 valeurs manquantes
_c13 : 154 valeurs manquantes
_c14 : 170 valeurs manquantes
_c15 : 158 valeurs manquantes
_c16 : 121 valeurs manquantes


In [22]:
from pyspark.sql import functions as F

df_web_sales.filter(
    F.col("_c10").isNull() |
    (F.trim(F.col("_c10")) == "")
).select(
    "_c0", "_c2", "_c4",
    "_c9", "_c10", "_c11",
    "_c13", "_c16"
).show(5, truncate=False)

+-----+-----+-----+-------+----+-----+----+----+
|_c0  |_c2  |_c4  |_c9    |_c10|_c11 |_c13|_c16|
+-----+-----+-----+-------+----+-----+----+----+
|36908|37002|75288|1787272|NULL|8581 |24  |253 |
|36961|37025|95479|481076 |NULL|14434|3   |142 |
|36961|37043|95479|481076 |NULL|14434|25  |148 |
|36961|37035|95479|481076 |NULL|14434|17  |141 |
|36961|37038|95479|481076 |NULL|14434|27  |143 |
+-----+-----+-----+-------+----+-----+----+----+
only showing top 5 rows



In [23]:
noms_web = [
    "ws_sold_date_sk",
    "ws_sold_time_sk",
    "ws_ship_date_sk",
    "ws_item_sk",
    "ws_bill_customer_sk",
    "ws_bill_cdemo_sk",
    "ws_bill_hdemo_sk",
    "ws_bill_addr_sk",
    "ws_ship_customer_sk",
    "ws_ship_cdemo_sk",
    "ws_ship_hdemo_sk",
    "ws_ship_addr_sk",
    "ws_web_page_sk",
    "ws_web_site_sk",
    "ws_ship_mode_sk",
    "ws_warehouse_sk",
    "ws_promo_sk",
    "ws_order_number",
    "ws_quantity",
    "ws_wholesale_cost",
    "ws_list_price",
    "ws_sales_price",
    "ws_ext_discount_amt",
    "ws_ext_sales_price",
    "ws_ext_wholesale_cost",
    "ws_ext_list_price",
    "ws_ext_tax",
    "ws_coupon_amt",
    "ws_ext_ship_cost",
    "ws_net_paid",
    "ws_net_paid_inc_tax",
    "ws_net_paid_inc_ship",
    "ws_net_paid_inc_ship_tax",
    "ws_net_profit"
]

assert len(noms_web) == len(df_web_sales.columns)

df_web_sales_nomme = df_web_sales.toDF(*noms_web)

df_web_sales_nomme.select(
    "ws_sold_date_sk",
    "ws_item_sk",
    "ws_quantity",
    "ws_net_paid",
    "ws_net_profit"
).show(5, truncate=False)

+---------------+----------+-----------+-----------+-------------+
|ws_sold_date_sk|ws_item_sk|ws_quantity|ws_net_paid|ws_net_profit|
+---------------+----------+-----------+-----------+-------------+
|36890          |794       |93         |5513.97    |2480.31      |
|36890          |9897      |60         |1047.42    |-160.98      |
|36890          |3725      |93         |12820.98   |8094.72      |
|36890          |8230      |38         |340.48     |-1153.30     |
|36890          |17454     |52         |178.82     |-2060.30     |
+---------------+----------+-----------+-----------+-------------+
only showing top 5 rows



In [24]:
from pyspark.sql import functions as F

# Les 19 premières colonnes : identifiants et quantité
colonnes_web_entieres = noms_web[:19]

# Les 15 dernières colonnes : prix et montants
colonnes_web_decimales = noms_web[19:]

# Créer un nouveau DataFrame pour les données typées
df_web_sales_type = df_web_sales_nomme

# Conversion des colonnes entières
for colonne in colonnes_web_entieres:
    df_web_sales_type = df_web_sales_type.withColumn(
        colonne,
        F.col(colonne).cast("integer")
    )

# Conversion des colonnes monétaires
for colonne in colonnes_web_decimales:
    df_web_sales_type = df_web_sales_type.withColumn(
        colonne,
        F.col(colonne).cast("decimal(18,2)")
    )

# Vérifier le nouveau schéma
df_web_sales_type.printSchema()

root
 |-- ws_sold_date_sk: integer (nullable = true)
 |-- ws_sold_time_sk: integer (nullable = true)
 |-- ws_ship_date_sk: integer (nullable = true)
 |-- ws_item_sk: integer (nullable = true)
 |-- ws_bill_customer_sk: integer (nullable = true)
 |-- ws_bill_cdemo_sk: integer (nullable = true)
 |-- ws_bill_hdemo_sk: integer (nullable = true)
 |-- ws_bill_addr_sk: integer (nullable = true)
 |-- ws_ship_customer_sk: integer (nullable = true)
 |-- ws_ship_cdemo_sk: integer (nullable = true)
 |-- ws_ship_hdemo_sk: integer (nullable = true)
 |-- ws_ship_addr_sk: integer (nullable = true)
 |-- ws_web_page_sk: integer (nullable = true)
 |-- ws_web_site_sk: integer (nullable = true)
 |-- ws_ship_mode_sk: integer (nullable = true)
 |-- ws_warehouse_sk: integer (nullable = true)
 |-- ws_promo_sk: integer (nullable = true)
 |-- ws_order_number: integer (nullable = true)
 |-- ws_quantity: integer (nullable = true)
 |-- ws_wholesale_cost: decimal(18,2) (nullable = true)
 |-- ws_list_price: decimal(18

In [25]:
from pyspark.sql import functions as F

controles_web_types = []

for colonne in df_web_sales_nomme.columns:

    if colonne in colonnes_web_entieres:
        type_cible = "integer"
    else:
        type_cible = "decimal(18,2)"

    valeur_source = F.col(colonne)
    valeur_convertie = valeur_source.cast(type_cible)

    erreur = F.sum(
        F.when(
            valeur_source.isNotNull()
            & (F.trim(valeur_source) != "")
            & valeur_convertie.isNull(),
            1
        ).otherwise(0)
    ).alias(colonne)

    controles_web_types.append(erreur)

resultat_types_web = df_web_sales_nomme.agg(
    *controles_web_types
)

# Afficher uniquement les colonnes avec des erreurs
resultats = resultat_types_web.first().asDict()

total_erreurs = sum(resultats.values())

print("Nombre total d'erreurs de conversion :", total_erreurs)

for colonne, nombre in resultats.items():
    if nombre > 0:
        print(f"{colonne} : {nombre} erreurs")

[Stage 50:============================>                            (5 + 5) / 10]

Nombre total d'erreurs de conversion : 0


In [26]:
# Nombre total de lignes
nb_total_web = df_web_sales_type.count()

# Nombre de lignes entièrement distinctes
nb_distinct_web = df_web_sales_type.distinct().count()

# Nombre de doublons exacts
nb_doublons_web = nb_total_web - nb_distinct_web

print("Nombre total de lignes :", nb_total_web)
print("Nombre de lignes distinctes :", nb_distinct_web)
print("Nombre de doublons exacts :", nb_doublons_web)

[Stage 56:=====>                                                   (1 + 9) / 10]

Nombre total de lignes : 668052
Nombre de lignes distinctes : 668052
Nombre de doublons exacts : 0


In [27]:
# Créer les deux DataFrames Silver
df_silver_store_sales = df_store_sales_type

df_silver_web_sales = df_web_sales_type

# Vérifier leur contenu
print("SILVER - STORE SALES")
print("Lignes :", df_silver_store_sales.count())
print("Colonnes :", len(df_silver_store_sales.columns))

print("\nSILVER - WEB SALES")
print("Lignes :", df_silver_web_sales.count())
print("Colonnes :", len(df_silver_web_sales.columns))

SILVER - STORE SALES
Lignes : 667579
Colonnes : 23

SILVER - WEB SALES
Lignes : 668052
Colonnes : 34


In [28]:
from pathlib import Path

# Dossier principal du TP
dossier_tp = Path.home() / "retail-intelligence/tp3"

# Dossiers de sortie Silver
chemin_silver_store = dossier_tp / "data/silver/store_sales"
chemin_silver_web = dossier_tp / "data/silver/web_sales"

# Enregistrer les ventes magasin au format Parquet
df_silver_store_sales.write \
    .mode("overwrite") \
    .parquet(str(chemin_silver_store))

# Enregistrer les ventes web au format Parquet
df_silver_web_sales.write \
    .mode("overwrite") \
    .parquet(str(chemin_silver_web))

print("Écriture Silver terminée.")
print("Store Sales :", chemin_silver_store)
print("Web Sales :", chemin_silver_web)

[Stage 69:=====>                                                   (1 + 9) / 10]

Écriture Silver terminée.
Store Sales : /Users/elkourafouad/retail-intelligence/tp3/data/silver/store_sales
Web Sales : /Users/elkourafouad/retail-intelligence/tp3/data/silver/web_sales


In [29]:
# Relire les fichiers Parquet depuis le disque
df_store_verification = spark.read.parquet(
    str(chemin_silver_store)
)

df_web_verification = spark.read.parquet(
    str(chemin_silver_web)
)

# Vérifier les volumes
print("STORE SALES")
print("Lignes :", df_store_verification.count())
print("Colonnes :", len(df_store_verification.columns))

print("\nWEB SALES")
print("Lignes :", df_web_verification.count())
print("Colonnes :", len(df_web_verification.columns))

# Vérifier les types de quelques colonnes
print("\nTYPES STORE SALES")
df_store_verification.select(
    "ss_quantity",
    "ss_net_paid",
    "ss_net_profit"
).printSchema()

print("\nTYPES WEB SALES")
df_web_verification.select(
    "ws_quantity",
    "ws_net_paid",
    "ws_net_profit"
).printSchema()

STORE SALES
Lignes : 667579
Colonnes : 23

WEB SALES
Lignes : 668052
Colonnes : 34

TYPES STORE SALES
root
 |-- ss_quantity: integer (nullable = true)
 |-- ss_net_paid: decimal(18,2) (nullable = true)
 |-- ss_net_profit: decimal(18,2) (nullable = true)


TYPES WEB SALES
root
 |-- ws_quantity: integer (nullable = true)
 |-- ws_net_paid: decimal(18,2) (nullable = true)
 |-- ws_net_profit: decimal(18,2) (nullable = true)



In [30]:
from pyspark.sql import functions as F

# Calculer les KPI des ventes en magasin
df_gold_store = df_store_verification.agg(
    F.sum("ss_quantity").alias("quantite_totale"),
    F.sum("ss_net_paid").alias("montant_net_total"),
    F.sum("ss_net_profit").alias("benefice_net_total")
)

# Ajouter le canal de vente
df_gold_store = df_gold_store.withColumn(
    "canal",
    F.lit("Magasin")
)

# Afficher les résultats
df_gold_store.show(truncate=False)

+---------------+-----------------+------------------+-------+
|quantite_totale|montant_net_total|benefice_net_total|canal  |
+---------------+-----------------+------------------+-------+
|33755474       |1150529586.37    |-552553374.82     |Magasin|
+---------------+-----------------+------------------+-------+



In [31]:
from pyspark.sql import functions as F

# Calculer les KPI des ventes en ligne
df_gold_web = df_web_verification.agg(
    F.sum("ws_quantity").alias("quantite_totale"),
    F.sum("ws_net_paid").alias("montant_net_total"),
    F.sum("ws_net_profit").alias("benefice_net_total")
)

# Identifier le canal de vente
df_gold_web = df_gold_web.withColumn(
    "canal",
    F.lit("Web")
)

# Afficher les résultats
df_gold_web.show(truncate=False)

+---------------+-----------------+------------------+-----+
|quantite_totale|montant_net_total|benefice_net_total|canal|
+---------------+-----------------+------------------+-----+
|33724378       |1535374825.67    |-168576494.45     |Web  |
+---------------+-----------------+------------------+-----+



In [32]:
# Réunir les résultats Magasin et Web
df_gold_multicanal = df_gold_store.unionByName(
    df_gold_web
)

# Afficher le tableau Gold final
df_gold_multicanal.select(
    "canal",
    "quantite_totale",
    "montant_net_total",
    "benefice_net_total"
).show(truncate=False)

+-------+---------------+-----------------+------------------+
|canal  |quantite_totale|montant_net_total|benefice_net_total|
+-------+---------------+-----------------+------------------+
|Magasin|33755474       |1150529586.37    |-552553374.82     |
|Web    |33724378       |1535374825.67    |-168576494.45     |
+-------+---------------+-----------------+------------------+



In [33]:
from pathlib import Path

# Chemin de sortie Gold
chemin_gold = (
    Path.home()
    / "retail-intelligence/tp3/data/gold/kpi_multicanal"
)

# Enregistrer les KPI au format Parquet
df_gold_multicanal.write \
    .mode("overwrite") \
    .parquet(str(chemin_gold))

print("Écriture Gold terminée.")
print("Emplacement :", chemin_gold)

Écriture Gold terminée.
Emplacement : /Users/elkourafouad/retail-intelligence/tp3/data/gold/kpi_multicanal


In [34]:
# Relire Gold depuis les fichiers Parquet
df_gold_verification = spark.read.parquet(
    str(chemin_gold)
)

# Vérifier le nombre de lignes et de colonnes
print("Nombre de lignes :", df_gold_verification.count())
print("Nombre de colonnes :", len(df_gold_verification.columns))

# Afficher le schéma
print("\nSchéma Gold :")
df_gold_verification.printSchema()

# Afficher les KPI
print("\nRésultats Gold :")
df_gold_verification.select(
    "canal",
    "quantite_totale",
    "montant_net_total",
    "benefice_net_total"
).show(truncate=False)


Nombre de lignes : 2
Nombre de colonnes : 4

Schéma Gold :
root
 |-- quantite_totale: long (nullable = true)
 |-- montant_net_total: decimal(28,2) (nullable = true)
 |-- benefice_net_total: decimal(28,2) (nullable = true)
 |-- canal: string (nullable = true)


Résultats Gold :
+-------+---------------+-----------------+------------------+
|canal  |quantite_totale|montant_net_total|benefice_net_total|
+-------+---------------+-----------------+------------------+
|Magasin|33755474       |1150529586.37    |-552553374.82     |
|Web    |33724378       |1535374825.67    |-168576494.45     |
+-------+---------------+-----------------+------------------+



In [35]:
donnees_qualite = [
    ("store_sales", 667579, 23, 4, 0, 0),
    ("web_sales", 668052, 34, 11, 0, 0)
]

colonnes_qualite = [
    "source",
    "nombre_lignes",
    "nombre_colonnes",
    "colonnes_avec_null",
    "erreurs_conversion",
    "doublons_exacts"
]

df_rapport_qualite = spark.createDataFrame(
    donnees_qualite,
    colonnes_qualite
)

df_rapport_qualite.show(truncate=False)

+-----------+-------------+---------------+------------------+------------------+---------------+
|source     |nombre_lignes|nombre_colonnes|colonnes_avec_null|erreurs_conversion|doublons_exacts|
+-----------+-------------+---------------+------------------+------------------+---------------+
|store_sales|667579       |23             |4                 |0                 |0              |
|web_sales  |668052       |34             |11                |0                 |0              |
+-----------+-------------+---------------+------------------+------------------+---------------+



In [36]:
from pathlib import Path

chemin_rapport = (
    Path.home()
    / "retail-intelligence/tp3/reports/qualite_sources"
)

df_rapport_qualite.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(str(chemin_rapport))

print("Rapport qualité CSV enregistré.")
print("Emplacement :", chemin_rapport)

Rapport qualité CSV enregistré.
Emplacement : /Users/elkourafouad/retail-intelligence/tp3/reports/qualite_sources


In [1]:
from pyspark.sql import functions as F

for source, df, colonne in [
    ("store_sales", df_store_verification, "ss_quantity"),
    ("web_sales", df_web_verification, "ws_quantity")
]:
    print(f"\nContrôle des quantités - {source}")

    df.agg(
        F.sum(
            F.when(F.col(colonne) <= 0, 1).otherwise(0)
        ).alias("quantites_nulles_ou_negatives"),

        F.min(colonne).alias("quantite_minimale"),
        F.max(colonne).alias("quantite_maximale")
    ).show()

NameError: name 'df_store_verification' is not defined

In [2]:
donnees_qualite = [
    ("store_sales", 667579, 23, 4, 0, 0),
    ("web_sales", 668052, 34, 11, 0, 0)
]

colonnes_qualite = [
    "source",
    "nombre_lignes",
    "nombre_colonnes",
    "colonnes_avec_null",
    "erreurs_conversion",
    "doublons_exacts"
]

df_rapport_qualite = spark.createDataFrame(
    donnees_qualite,
    colonnes_qualite
)

df_rapport_qualite.show(truncate=False)

NameError: name 'spark' is not defined

In [3]:
donnees_qualite = [
    ("store_sales", 667579, 23, 4, 0, 0),
    ("web_sales", 668052, 34, 11, 0, 0)
]

colonnes_qualite = [
    "source",
    "nombre_lignes",
    "nombre_colonnes",
    "colonnes_avec_null",
    "erreurs_conversion",
    "doublons_exacts"
]

df_rapport_qualite = spark.createDataFrame(
    donnees_qualite,
    colonnes_qualite
)

df_rapport_qualite.show(truncate=False)

NameError: name 'spark' is not defined

In [4]:
from pyspark.sql import functions as F

for source, df, colonne in [
    ("store_sales", df_store_verification, "ss_quantity"),
    ("web_sales", df_web_verification, "ws_quantity")
]:
    print(f"\nContrôle des quantités - {source}")

    df.agg(
        F.sum(
            F.when(F.col(colonne) <= 0, 1).otherwise(0)
        ).alias("quantites_nulles_ou_negatives"),
        F.min(colonne).alias("quantite_minimale"),
        F.max(colonne).alias("quantite_maximale")
    ).show()

NameError: name 'df_store_verification' is not defined

In [4]:
from pyspark.sql import functions as F

for source, df, colonne in [
    ("store_sales", df_store_verification, "ss_quantity"),
    ("web_sales", df_web_verification, "ws_quantity")
]:
    print(f"\nContrôle des quantités - {source}")

    df.agg(
        F.sum(
            F.when(F.col(colonne) <= 0, 1).otherwise(0)
        ).alias("quantites_nulles_ou_negatives"),
        F.min(colonne).alias("quantite_minimale"),
        F.max(colonne).alias("quantite_maximale")
    ).show()


Contrôle des quantités - store_sales
+-----------------------------+-----------------+-----------------+
|quantites_nulles_ou_negatives|quantite_minimale|quantite_maximale|
+-----------------------------+-----------------+-----------------+
|                            0|                1|              100|
+-----------------------------+-----------------+-----------------+


Contrôle des quantités - web_sales
+-----------------------------+-----------------+-----------------+
|quantites_nulles_ou_negatives|quantite_minimale|quantite_maximale|
+-----------------------------+-----------------+-----------------+
|                            0|                1|              100|
+-----------------------------+-----------------+-----------------+



In [5]:
from pyspark.sql import functions as F

df_web_verification.agg(
    F.sum(
        F.when(
            F.col("ws_ship_date_sk") <
            F.col("ws_sold_date_sk"),
            1
        ).otherwise(0)
    ).alias("expeditions_avant_vente"),

    F.sum(
        F.when(
            F.col("ws_ship_date_sk").isNull(),
            1
        ).otherwise(0)
    ).alias("dates_expedition_manquantes"),

    F.min("ws_sold_date_sk").alias("date_vente_min"),
    F.max("ws_sold_date_sk").alias("date_vente_max")

).show(truncate=False)

+-----------------------+---------------------------+--------------+--------------+
|expeditions_avant_vente|dates_expedition_manquantes|date_vente_min|date_vente_max|
+-----------------------+---------------------------+--------------+--------------+
|0                      |0                          |36890         |38697         |
+-----------------------+---------------------------+--------------+--------------+



In [6]:
from pyspark.sql import functions as F

sources = [
    ("store_sales", df_store_verification, "ss_net_paid", "ss_net_profit"),
    ("web_sales", df_web_verification, "ws_net_paid", "ws_net_profit")
]

for nom, df, montant, benefice in sources:

    print(f"\nContrôle financier : {nom}")

    df.agg(
        F.sum(
            F.when(F.col(montant) < 0, 1).otherwise(0)
        ).alias("montants_negatifs"),

        F.sum(
            F.when(F.col(montant) == 0, 1).otherwise(0)
        ).alias("montants_nuls"),

        F.sum(
            F.when(F.col(benefice) < 0, 1).otherwise(0)
        ).alias("benefices_negatifs"),

        F.min(montant).alias("montant_minimum"),
        F.max(montant).alias("montant_maximum")
    ).show(truncate=False)


Contrôle financier : store_sales
+-----------------+-------------+------------------+---------------+---------------+
|montants_negatifs|montants_nuls|benefices_negatifs|montant_minimum|montant_maximum|
+-----------------+-------------+------------------+---------------+---------------+
|0                |3954         |495971            |0.00           |18745.00       |
+-----------------+-------------+------------------+---------------+---------------+


Contrôle financier : web_sales
+-----------------+-------------+------------------+---------------+---------------+
|montants_negatifs|montants_nuls|benefices_negatifs|montant_minimum|montant_maximum|
+-----------------+-------------+------------------+---------------+---------------+
|0                |3938         |407163            |0.00           |28808.01       |
+-----------------+-------------+------------------+---------------+---------------+



In [7]:
df_store_verification.filter(
    F.col("ss_net_paid") == 0
).select(
    "ss_quantity",
    "ss_sales_price",
    "ss_ext_discount_amt",
    "ss_coupon_amt",
    "ss_net_paid",
    "ss_net_profit"
).show(10, truncate=False)

+-----------+--------------+-------------------+-------------+-----------+-------------+
|ss_quantity|ss_sales_price|ss_ext_discount_amt|ss_coupon_amt|ss_net_paid|ss_net_profit|
+-----------+--------------+-------------------+-------------+-----------+-------------+
|34         |0.00          |220.66             |0.00         |0.00       |-171.02      |
|99         |0.00          |2987.82            |0.00         |0.00       |-1632.51     |
|10         |0.00          |1436.40            |0.00         |0.00       |-736.60      |
|95         |0.00          |7783.35            |0.00         |0.00       |-3931.10     |
|3          |0.00          |227.13             |0.00         |0.00       |-115.29      |
|11         |0.00          |864.82             |0.00         |0.00       |-726.77      |
|17         |0.00          |1029.69            |0.00         |0.00       |-591.77      |
|95         |33.69         |902.50             |3200.55      |0.00       |-2909.85     |
|49         |0.00    

In [8]:
from pyspark.sql import functions as F

df_store_verification.agg(
    F.sum(
        F.when(
            F.col("ss_net_paid") !=
            (
                F.col("ss_ext_sales_price")
                - F.col("ss_coupon_amt")
            ),
            1
        ).otherwise(0)
    ).alias("incoherences_montant"),

    F.sum(
        F.when(
            F.col("ss_net_paid").isNull()
            | F.col("ss_ext_sales_price").isNull()
            | F.col("ss_coupon_amt").isNull(),
            1
        ).otherwise(0)
    ).alias("lignes_non_verifiables")
).show()

+--------------------+----------------------+
|incoherences_montant|lignes_non_verifiables|
+--------------------+----------------------+
|                   0|                     0|
+--------------------+----------------------+



In [9]:
from pyspark.sql import functions as F

df_web_verification.agg(
    F.sum(
        F.when(
            F.col("ws_net_paid") !=
            (
                F.col("ws_ext_sales_price")
                - F.col("ws_coupon_amt")
            ),
            1
        ).otherwise(0)
    ).alias("incoherences_montant"),

    F.sum(
        F.when(
            F.col("ws_net_paid").isNull()
            | F.col("ws_ext_sales_price").isNull()
            | F.col("ws_coupon_amt").isNull(),
            1
        ).otherwise(0)
    ).alias("lignes_non_verifiables")
).show()

+--------------------+----------------------+
|incoherences_montant|lignes_non_verifiables|
+--------------------+----------------------+
|                   0|                     0|
+--------------------+----------------------+



In [10]:
from pathlib import Path

chemin_item = (
    Path.home()
    / "Downloads/retail_project/data/item/item.dat"
)

df_item_raw = (
    spark.read
    .option("header", False)
    .option("sep", "|")
    .csv(str(chemin_item))
)

print("Nombre de lignes :", df_item_raw.count())
print("Nombre de colonnes :", len(df_item_raw.columns))

df_item_raw.printSchema()

df_item_raw.show(3, truncate=30)


ConnectionRefusedError: [Errno 61] Connection refused

In [11]:
from pathlib import Path

chemin_item = (
    Path.home()
    / "Downloads/retail_project/data/item/item.dat"
)

df_item_raw = (
    spark.read
    .option("header", False)
    .option("sep", "|")
    .csv(str(chemin_item))
)

print("Nombre de lignes :", df_item_raw.count())
print("Nombre de colonnes :", len(df_item_raw.columns))

df_item_raw.printSchema()

df_item_raw.show(3, truncate=30)

ConnectionRefusedError: [Errno 61] Connection refused